# Supplementary Experiment: Directional PAE Edge Features
A second, independent robustness check on the project's central negative result (PAE-weighting via `1/(1+PAE)` does not beat an unweighted GATv2 ablation). This notebook tests a **genuinely new source of information**, not another rescaling of the same number: AlphaFold's PAE matrix is not symmetric, and the existing pipeline throws away that asymmetry by averaging `PAE[i][j]` and `PAE[j][i]` into one scalar before anything else happens. This notebook keeps both directions as separate edge features instead.

**Why this is different from the min-max normalization experiment (`pae_minmax_experiment/`):** that experiment showed per-protein standard deviation and the fraction of tied edges were *identical* between the original transform and min-max normalization, because both are monotonic reparametrizations of the same underlying number, and a monotonic function cannot un-tie two edges that already had the same PAE. This experiment is not a reparametrization, it adds a feature dimension the model never had access to before.

### The diagnostic that motivates this
On the same 300-protein sample used throughout this project: every edge sampled was tied with at least one other edge in its protein on the symmetrized PAE value (a near-inevitable consequence of averaging + AlphaFold's coarse ~1 Å quantization). Of those tied edges, **32.8%** actually have `PAE[i][j] != PAE[j][i]`, meaning the *raw* AlphaFold output already distinguishes them; the averaging step in the existing pipeline is what erases that distinction, not a fundamental data limit. The other 67.2% of ties are genuine, AlphaFold really did report identical confidence in both directions, so this is a **partial**, honestly-scoped fix, not a claim that it resolves the whole saturation problem.

### The change
```
OLD edge_attr (2-dim): [distance, 1/(1 + average(PAE[i][j], PAE[j][i]))]
NEW edge_attr (3-dim): [distance, 1/(1 + PAE[src][dst]), 1/(1 + PAE[dst][src])]
```
`GATv2Conv`'s `edge_dim` parameter is simply set to 3 instead of 2, `models.py` already supports this via its existing `edge_dim` constructor argument, so **no changes to the shared `models.py` or `missense_dataset.py` files are needed.**

**Does not modify any existing notebook, checkpoint, or result file.** Every output is written under `pae_directional_experiment/`. All existing project files (`protein_graphs/`, `models/`, `phase4_*.csv`, `phase5_*.csv`, `clinvar_graph_ready.csv`, `best_config.json`) are only ever **read**.

**Inputs (all read-only):**
- `clinvar_graph_ready.csv`, `protein_graphs/{UniProt_ID}.pt`, topology and node features reused unchanged; only `edge_attr` is rebuilt with a 3rd column
- `alphafold_data/pae/{UniProt_ID}.json`, raw PAE matrices (already cached from Phase 2)
- `best_config.json`, `models/gatv2_unweighted_best.pt`, `models/gatv2_unweighted_final_best.pt`, `phase5_gnn_predictions.csv`, `alphamissense_test_scores.csv`, reused exactly as in the min-max experiment, for the same reasons (unweighted ablation never reads `edge_attr` regardless of its shape, so retraining it again would waste ~14h reproducing known numbers)

**Outputs (all new, under `pae_directional_experiment/`):**
- `protein_graphs_directional/{UniProt_ID}.pt`, 6,597 rebuilt graphs, 3-dim edge_attr
- `models/gatv2_pae_directional_default_best.pt`, `models/gatv2_pae_directional_tuned_best.pt`
- `training_history_*.csv`, `figures/training_curves_*.png`
- `tie_breaking_diagnostic.csv`, proof of how many ties this actually breaks, checked before training
- `results_comparison.csv`, `significance_tests.csv`, `pae_value_quantification_directional.csv`
- `alphamissense_comparison_results.csv`, `alphamissense_significance_tests.csv`, `figures/alphamissense_comparison_directional.png`
- `predictions_cache.csv`


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import Sampler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from scipy import stats
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                              recall_score, f1_score, accuracy_score)
from statsmodels.stats.contingency_tables import mcnemar

from missense_dataset import MissenseVariantGraphDataset, NODE_FEATURE_DIM
from models import GATv2Classifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# --- Read-only sources from the existing project (never written to) ---
OLD_GRAPH_DIR   = "protein_graphs"
OLD_MODELS_DIR  = "models"
PAE_DIR         = os.path.join("alphafold_data", "pae")
BEST_CONFIG_PATH = "best_config.json"
PHASE5_PRED_CACHE = "phase5_gnn_predictions.csv"
AM_CACHE_PATH = "alphamissense_test_scores.csv"

# --- New, isolated output tree for this experiment only ---
EXPERIMENT_DIR = "pae_directional_experiment"
NEW_GRAPH_DIR  = os.path.join(EXPERIMENT_DIR, "protein_graphs_directional")
NEW_MODELS_DIR = os.path.join(EXPERIMENT_DIR, "models")
NEW_FIGURES_DIR = os.path.join(EXPERIMENT_DIR, "figures")
for d in (EXPERIMENT_DIR, NEW_GRAPH_DIR, NEW_MODELS_DIR, NEW_FIGURES_DIR):
    os.makedirs(d, exist_ok=True)

EDGE_DIM = 3  # [distance, forward-direction weight, reverse-direction weight]

torch.manual_seed(42)
np.random.seed(42)


## 1. Load Phase 3 Output (read-only)


In [ ]:
df = pd.read_csv("clinvar_graph_ready.csv")
print(f"Total variants : {len(df):,}")
print(f"Unique proteins: {df['UniProt_ID'].nunique():,}")
print(f"  Pathogenic (1): {(df['Target']==1).sum():,}")
print(f"  Benign     (0): {(df['Target']==0).sum():,}")


## 2. Rebuild All Protein Graphs With Directional PAE Features
Reuses each protein's cached topology, node features, and Cα-Cα distances unchanged, only `edge_attr` grows from 2 columns to 3: `[distance, 1/(1+PAE[src][dst]), 1/(1+PAE[dst][src])]`. For a directed arc `(src, dst)`, `PAE[src][dst]` is that arc's own directional confidence and `PAE[dst][src]` is the reverse direction's, both stored explicitly rather than averaged away. No PDB re-parsing, no ESM-2 recomputation. Resumable exactly like Phase 2/3 and the min-max experiment.


In [ ]:
def rebuild_graph_directional(uniprot_id):
    out_path = os.path.join(NEW_GRAPH_DIR, f"{uniprot_id}.pt")
    if os.path.exists(out_path):
        return "cached"
    old_path = os.path.join(OLD_GRAPH_DIR, f"{uniprot_id}.pt")
    pae_path = os.path.join(PAE_DIR, f"{uniprot_id}.json")
    if not (os.path.exists(old_path) and os.path.exists(pae_path)):
        return "missing_source_file"

    g = torch.load(old_path, weights_only=False)
    with open(pae_path) as f:
        pae_matrix = np.array(json.load(f)[0]["predicted_aligned_error"], dtype=np.float32)
    if pae_matrix.shape[0] != g.x.shape[0]:
        return "pae_shape_mismatch"

    src, dst = g.edge_index.numpy()
    weight_fwd = 1.0 / (1.0 + pae_matrix[src, dst])
    weight_rev = 1.0 / (1.0 + pae_matrix[dst, src])
    distances = g.edge_attr[:, 0].numpy()  # unchanged

    new_edge_attr = torch.tensor(
        np.stack([distances, weight_fwd.astype(np.float32), weight_rev.astype(np.float32)], axis=1),
        dtype=torch.float)

    new_g = Data(x=g.x, edge_index=g.edge_index, edge_attr=new_edge_attr,
                 uniprot_id=uniprot_id, num_residues=g.num_residues)
    torch.save(new_g, out_path)
    return "success"


unique_proteins = df["UniProt_ID"].unique()
rebuild_status = {}
for uid in tqdm(unique_proteins, desc="Rebuilding graphs (directional PAE)"):
    rebuild_status[uid] = rebuild_graph_directional(uid)

status_counts = pd.Series(rebuild_status).value_counts()
print("\nGraph rebuild status:")
print(status_counts.to_string())
failed = {k: v for k, v in rebuild_status.items() if v not in ("cached", "success")}
if failed:
    print(f"\nWARNING: {len(failed)} proteins failed to rebuild.")


## 3. Diagnostic: How Many Ties Does This Actually Break?
Checked **before** the ~14 hour training commitment, on the same 300-protein sample (seed=0) used throughout this project, so the numbers below are directly comparable to the earlier saturation and min-max diagnostics.


In [ ]:
rng_diag = np.random.RandomState(0)
sample_proteins_diag = rng_diag.choice(df["UniProt_ID"].unique(), size=300, replace=False)

n_edges_total = 0
n_tied_on_avg = 0
n_tied_broken_by_direction = 0

for uid in tqdm(sample_proteins_diag, desc="Checking tie-breaking"):
    old_g = torch.load(os.path.join(OLD_GRAPH_DIR, f"{uid}.pt"), weights_only=False)
    new_g = torch.load(os.path.join(NEW_GRAPH_DIR, f"{uid}.pt"), weights_only=False)
    pae_sym = old_g.edge_attr[:, 1].numpy()  # old symmetrized weight, used only to find ties
    weight_fwd = new_g.edge_attr[:, 1].numpy()
    weight_rev = new_g.edge_attr[:, 2].numpy()

    n_edges_total += len(pae_sym)
    vals, counts = np.unique(pae_sym, return_counts=True)
    tied_mask = np.isin(pae_sym, vals[counts > 1])
    n_tied_on_avg += tied_mask.sum()
    directional_differs = np.abs(weight_fwd - weight_rev) > 1e-6
    n_tied_broken_by_direction += (tied_mask & directional_differs).sum()

frac_tied = n_tied_on_avg / n_edges_total
frac_broken = n_tied_broken_by_direction / max(n_tied_on_avg, 1)
print(f"Sampled {len(sample_proteins_diag)} proteins, {n_edges_total:,} total edges")
print(f"Edges tied under the old symmetrized transform: {n_tied_on_avg:,} ({frac_tied:.1%})")
print(f"Of those, fraction now distinguishable via directional features: {frac_broken:.1%}")
print(f"(Expect roughly a third broken, two-thirds genuinely tied even with full directional info "
      f"- this is a partial fix, not a complete one, per the earlier diagnostic.)")

pd.DataFrame([{"n_edges_sampled": n_edges_total, "n_tied_old": n_tied_on_avg,
               "frac_tied_old": frac_tied, "n_tied_broken_by_direction": n_tied_broken_by_direction,
               "frac_of_ties_broken": frac_broken}]
             ).to_csv(os.path.join(EXPERIMENT_DIR, "tie_breaking_diagnostic.csv"), index=False)


## 4. Train/Val/Test Split, Batching, Class Weighting
Identical logic and seed (42) to Phase 4/5 and the min-max experiment, reproduced here so this notebook has zero import-time dependency on any other notebook having been run.


In [ ]:
def make_protein_split(df, seed, train_frac=0.70, val_frac=0.15):
    rng = np.random.RandomState(seed)
    unique_proteins = df["UniProt_ID"].unique().copy()
    rng.shuffle(unique_proteins)
    n = len(unique_proteins)
    n_train, n_val = int(n * train_frac), int(n * val_frac)
    train_p = set(unique_proteins[:n_train])
    val_p   = set(unique_proteins[n_train:n_train + n_val])
    test_p  = set(unique_proteins[n_train + n_val:])
    assert train_p.isdisjoint(val_p) and train_p.isdisjoint(test_p) and val_p.isdisjoint(test_p)
    return (df[df["UniProt_ID"].isin(p)].reset_index(drop=True) for p in (train_p, val_p, test_p))


train_df, val_df, test_df = make_protein_split(df, seed=42)
print("Protein-level split is leak-free (verified disjoint by assertion above).")
for name, sub in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    frac = (sub["Target"] == 1).mean()
    print(f"{name:5s}: {len(sub):6,} variants, {sub['UniProt_ID'].nunique():4,} proteins, "
          f"{frac:.3f} pathogenic fraction")


In [ ]:
class NodeBudgetBatchSampler(Sampler):
    def __init__(self, node_counts, max_nodes_per_batch=20000, shuffle=True, seed=0):
        self.node_counts = node_counts
        self.max_nodes_per_batch = max_nodes_per_batch
        self.shuffle = shuffle
        self.seed = seed
        self._epoch = 0

    def __iter__(self):
        indices = list(range(len(self.node_counts)))
        if self.shuffle:
            r = random.Random(self.seed + self._epoch)
            r.shuffle(indices)
            self._epoch += 1
        batch, current = [], 0
        for idx in indices:
            n = self.node_counts[idx]
            if batch and current + n > self.max_nodes_per_batch:
                yield batch
                batch, current = [], 0
            batch.append(idx)
            current += n
        if batch:
            yield batch

    def __len__(self):
        return max(1, sum(self.node_counts) // self.max_nodes_per_batch)


def get_node_counts(variant_df, graph_dir):
    cache, counts = {}, []
    for uid in variant_df["UniProt_ID"]:
        if uid not in cache:
            cache[uid] = torch.load(os.path.join(graph_dir, f"{uid}.pt"), weights_only=False).num_residues
        counts.append(cache[uid])
    return counts


train_ds = MissenseVariantGraphDataset(train_df, NEW_GRAPH_DIR)
val_ds   = MissenseVariantGraphDataset(val_df, NEW_GRAPH_DIR)
test_ds  = MissenseVariantGraphDataset(test_df, NEW_GRAPH_DIR)

train_loader = DataLoader(train_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(train_df, NEW_GRAPH_DIR), 20000, shuffle=True, seed=42))
val_loader = DataLoader(val_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(val_df, NEW_GRAPH_DIR), 20000, shuffle=False))
test_loader = DataLoader(test_ds, batch_sampler=NodeBudgetBatchSampler(
    get_node_counts(test_df, NEW_GRAPH_DIR), 20000, shuffle=False))

train_pos = (train_df["Target"] == 1).sum()
train_neg = (train_df["Target"] == 0).sum()
pos_weight_value = train_neg / train_pos
print(f"pos_weight = {pos_weight_value:.3f}")
POS_WEIGHT = torch.tensor([pos_weight_value], dtype=torch.float, device=DEVICE)


## 5. Training and Evaluation Functions
Identical methodology to Phase 4 and the min-max experiment (`run_epoch`, `train_model`, `full_metrics`, `evaluate_gnn`), same loss, same AUPRC-driven early stopping (patience=10, up to 60 epochs), same resumability contract.


In [ ]:
def run_epoch(model, loader, pos_weight, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    losses, all_logits, all_labels = [], [], []
    with torch.set_grad_enabled(is_train):
        for batch in loader:
            batch = batch.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            logits = model(batch)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, batch.y, pos_weight=pos_weight)
            if is_train:
                loss.backward()
                optimizer.step()
            losses.append(loss.item())
            all_logits.extend(logits.detach().cpu().numpy().tolist())
            all_labels.extend(batch.y.detach().cpu().numpy().tolist())
    auprc = average_precision_score(all_labels, all_logits)
    auroc = roc_auc_score(all_labels, all_logits)
    return float(np.mean(losses)), auprc, auroc


def train_model(model, model_name, train_loader, val_loader, pos_weight, n_epochs=60, patience=10,
                lr=1e-3, weight_decay=1e-5, verbose=True, resume=True):
    checkpoint_path = os.path.join(NEW_MODELS_DIR, f"{model_name}_best.pt")
    history_path = os.path.join(EXPERIMENT_DIR, f"training_history_{model_name}.csv")

    if resume and os.path.exists(checkpoint_path) and os.path.exists(history_path):
        model.to(DEVICE)
        model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
        if verbose:
            print(f"[{model_name}] Found existing checkpoint - skipping training and reloading it. "
                  f"Delete {checkpoint_path} first if you want to retrain from scratch.")
        history_df = pd.read_csv(history_path)
        return model, history_df, history_df["val_auprc"].max()

    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_auprc = -1.0
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, n_epochs + 1):
        train_loss, train_auprc, train_auroc = run_epoch(model, train_loader, pos_weight, optimizer)
        val_loss, val_auprc, val_auroc = run_epoch(model, val_loader, pos_weight, optimizer=None)
        history.append({"epoch": epoch, "train_loss": train_loss, "train_auprc": train_auprc,
                         "train_auroc": train_auroc, "val_loss": val_loss, "val_auprc": val_auprc,
                         "val_auroc": val_auroc})
        if verbose:
            print(f"[{model_name}] epoch {epoch:2d}: train_loss={train_loss:.4f} train_AUPRC={train_auprc:.3f} "
                  f"| val_loss={val_loss:.4f} val_AUPRC={val_auprc:.3f} val_AUROC={val_auroc:.3f}")
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    print(f"[{model_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)
    del optimizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return model, history_df, best_val_auprc


def plot_training_curves(history_df, model_name, safe_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train loss")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title(f"{model_name}: Loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["train_auprc"], label="train AUPRC")
    axes[1].plot(history_df["epoch"], history_df["val_auprc"], label="val AUPRC")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUPRC"); axes[1].set_title(f"{model_name}: AUPRC")
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(os.path.join(NEW_FIGURES_DIR, f"training_curves_{safe_name}.png"), dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def full_metrics(labels, probs, threshold=0.5):
    preds = (np.asarray(probs) >= threshold).astype(int)
    return {
        "AUPRC": average_precision_score(labels, probs),
        "AUROC": roc_auc_score(labels, probs),
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
    }


def evaluate_gnn(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            all_logits.extend(logits.cpu().numpy().tolist())
            all_labels.extend(batch.y.cpu().numpy().tolist())
    probs = torch.sigmoid(torch.tensor(all_logits)).numpy()
    return np.array(all_labels), probs


## 6. Train GATv2 (Directional PAE), Default Hyperparameters
Same defaults as the project's original pass: `hidden_dim=128, heads=4, dropout=0.2, lr=1e-3, weight_decay=1e-5`. `edge_dim=3` is the only architectural difference from the original PAE-weighted model.


In [ ]:
model_directional_default = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=True, edge_dim=EDGE_DIM)
model_directional_default, history_directional_default, _ = train_model(
    model_directional_default, "gatv2_pae_directional_default", train_loader, val_loader, POS_WEIGHT)
plot_training_curves(history_directional_default, "GATv2 (directional PAE, default)", "gatv2_pae_directional_default")


## 7. Train GATv2 (Directional PAE), Tuned `B_reg_small` Config
Reuses the project's existing winning hyperparameters from `best_config.json` (read-only).


In [ ]:
with open(BEST_CONFIG_PATH) as f:
    best_config = json.load(f)
print(f"Loaded best_config: {best_config['name']}  hidden_dim={int(best_config['hidden_dim'])}  "
      f"dropout={best_config['dropout']:.2f}  lr={best_config['lr']}  wd={best_config['weight_decay']}")

model_directional_tuned = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=True, edge_dim=EDGE_DIM,
                                           hidden_dim=int(best_config["hidden_dim"]), dropout=best_config["dropout"])
model_directional_tuned, history_directional_tuned, _ = train_model(
    model_directional_tuned, "gatv2_pae_directional_tuned", train_loader, val_loader, POS_WEIGHT,
    lr=best_config["lr"], weight_decay=best_config["weight_decay"])
plot_training_curves(history_directional_tuned, "GATv2 (directional PAE, tuned)", "gatv2_pae_directional_tuned")


## 8. Reload the Existing Unweighted Ablation (No Retraining)
Same reasoning as the min-max experiment: `use_edge_attr=False` means `forward()` never reads `edge_attr` regardless of its shape (2-dim or 3-dim), so the unweighted ablation's predictions are provably unaffected by this change. Its existing checkpoints are reloaded for inference only.


In [ ]:
model_unweighted_default = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=False)
model_unweighted_default.load_state_dict(
    torch.load(os.path.join(OLD_MODELS_DIR, "gatv2_unweighted_best.pt"), weights_only=True))
model_unweighted_default.to(DEVICE)

model_unweighted_tuned = GATv2Classifier(in_dim=NODE_FEATURE_DIM, use_edge_attr=False,
                                          hidden_dim=int(best_config["hidden_dim"]), dropout=best_config["dropout"])
model_unweighted_tuned.load_state_dict(
    torch.load(os.path.join(OLD_MODELS_DIR, "gatv2_unweighted_final_best.pt"), weights_only=True))
model_unweighted_tuned.to(DEVICE)

print("Reloaded both existing unweighted checkpoints (no training performed).")


## 9. Evaluate All Models on the Test Set


In [ ]:
test_labels, probs_directional_default = evaluate_gnn(model_directional_default, test_loader)
test_labels_2, probs_directional_tuned = evaluate_gnn(model_directional_tuned, test_loader)
test_labels_3, probs_unweighted_default = evaluate_gnn(model_unweighted_default, test_loader)
test_labels_4, probs_unweighted_tuned = evaluate_gnn(model_unweighted_tuned, test_loader)
assert (np.array_equal(test_labels, test_labels_2) and np.array_equal(test_labels, test_labels_3)
        and np.array_equal(test_labels, test_labels_4)), "Label misalignment across models - do not trust results"

test_df = test_df.reset_index(drop=True)
test_df["true_label"] = test_labels.astype(int)
test_df["directional_default_prob"] = probs_directional_default
test_df["directional_tuned_prob"] = probs_directional_tuned
test_df["unweighted_default_prob"] = probs_unweighted_default
test_df["unweighted_tuned_prob"] = probs_unweighted_tuned

pred_cache = pd.read_csv(PHASE5_PRED_CACHE)
test_df = test_df.merge(
    pred_cache[["VariationID", "gatv2_prob", "sage_prob"]].rename(
        columns={"gatv2_prob": "old_transform_tuned_prob", "sage_prob": "graphsage_prob"}),
    on="VariationID", how="left")

test_df.to_csv(os.path.join(EXPERIMENT_DIR, "predictions_cache.csv"), index=False)
print(f"Cached row-level predictions for {len(test_df):,} test variants.")


## 10. Combined Results Table


In [ ]:
new_rows = [
    {"model": "GATv2 (directional PAE, default)", **full_metrics(test_labels, probs_directional_default)},
    {"model": "GATv2 (directional PAE, tuned)", **full_metrics(test_labels, probs_directional_tuned)},
    {"model": "GATv2 (unweighted, default) [reloaded]", **full_metrics(test_labels, probs_unweighted_default)},
    {"model": "GATv2 (unweighted, tuned) [reloaded]", **full_metrics(test_labels, probs_unweighted_tuned)},
]

existing = pd.read_csv("phase4_results_final.csv")
for name in ["GraphSAGE (baseline)", "XGBoost (structure-free baseline)", "GCN (baseline)",
             "GATv2 (PAE-weighted, tuned)", "GATv2 (PAE-weighted, original)"]:
    row = existing[existing["model"] == name]
    if len(row):
        r = row.iloc[0].to_dict()
        r["model"] = r["model"] + " [existing, unchanged]"
        new_rows.append(r)

# Also pull in the min-max transform's tuned result for a 3-way comparison, if that experiment has been run
minmax_results_path = os.path.join("pae_minmax_experiment", "results_comparison.csv")
if os.path.exists(minmax_results_path):
    minmax_existing = pd.read_csv(minmax_results_path)
    row = minmax_existing[minmax_existing["model"] == "GATv2 (min-max PAE, tuned)"]
    if len(row):
        r = row.iloc[0].to_dict()
        r["model"] = r["model"] + " [existing, unchanged]"
        new_rows.append(r)

results_comparison_df = pd.DataFrame(new_rows).sort_values("AUPRC", ascending=False)
results_comparison_df.to_csv(os.path.join(EXPERIMENT_DIR, "results_comparison.csv"), index=False)
print(results_comparison_df.to_string(index=False))


## 11. Statistical Significance Testing
Identical methodology to Phase 4 and the min-max experiment.


In [ ]:
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2


def delong_test(probs_a, probs_b, labels):
    labels = np.asarray(labels)
    order = np.argsort(-labels)
    m = int(np.sum(labels == 1)); n = int(np.sum(labels == 0))
    preds_list = [np.asarray(probs_a), np.asarray(probs_b)]
    k = len(preds_list)
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        preds_ordered = preds_list[r][order]
        tx[r, :] = compute_midrank(preds_ordered[:m])
        ty[r, :] = compute_midrank(preds_ordered[m:])
        tz[r, :] = compute_midrank(preds_ordered)
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    delongcov = np.cov(v01) / m + np.cov(v10) / n
    assert abs(aucs[0] - roc_auc_score(labels, probs_a)) < 1e-6
    assert abs(aucs[1] - roc_auc_score(labels, probs_b)) < 1e-6
    l = np.array([1, -1])
    z = l.dot(aucs) / np.sqrt(l.dot(delongcov).dot(l))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"auc_a": aucs[0], "auc_b": aucs[1], "z": z, "p_value": p}


def paired_bootstrap_auprc(probs_a, probs_b, labels, n_boot=2000, seed=0):
    rng_b = np.random.RandomState(seed)
    labels = np.asarray(labels); probs_a = np.asarray(probs_a); probs_b = np.asarray(probs_b)
    n_test = len(labels)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng_b.randint(0, n_test, n_test)
        diffs[i] = (average_precision_score(labels[idx], probs_a[idx])
                    - average_precision_score(labels[idx], probs_b[idx]))
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    p_value = 2 * min((diffs > 0).mean(), (diffs < 0).mean())
    return {"mean_diff": diffs.mean(), "ci_low": ci_low, "ci_high": ci_high, "p_value": p_value}


def mcnemar_test(probs_a, probs_b, labels, threshold=0.5):
    preds_a = (np.asarray(probs_a) >= threshold).astype(int)
    preds_b = (np.asarray(probs_b) >= threshold).astype(int)
    correct_a = preds_a == labels; correct_b = preds_b == labels
    table = [[int(np.sum(correct_a & correct_b)), int(np.sum(correct_a & ~correct_b))],
             [int(np.sum(~correct_a & correct_b)), int(np.sum(~correct_a & ~correct_b))]]
    result = mcnemar(table, exact=False, correction=True)
    return {"statistic": result.statistic, "p_value": result.pvalue}


def run_all_significance_tests(name_a, probs_a, name_b, probs_b, labels):
    mc = mcnemar_test(probs_a, probs_b, labels)
    bs = paired_bootstrap_auprc(probs_a, probs_b, labels)
    dl = delong_test(probs_a, probs_b, labels)
    significant = bs["p_value"] < 0.05 or dl["p_value"] < 0.05
    print(f"\n=== {name_a}  vs.  {name_b} ===")
    print(f"McNemar p={mc['p_value']:.4f} | Bootstrap AUPRC diff={bs['mean_diff']:+.4f} "
          f"(p={bs['p_value']:.4f}) | DeLong AUROC p={dl['p_value']:.4f} "
          f"-> {'SIGNIFICANT' if significant else 'not significant'}")
    return {"comparison": f"{name_a} vs {name_b}", "mcnemar_p": mc["p_value"],
            "bootstrap_auprc_diff": bs["mean_diff"], "bootstrap_ci_low": bs["ci_low"],
            "bootstrap_ci_high": bs["ci_high"], "bootstrap_p": bs["p_value"],
            "delong_z": dl["z"], "delong_p": dl["p_value"], "significant": significant}


sig_rows = []
sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, default)", probs_directional_default,
    "GATv2 (unweighted, default)", probs_unweighted_default, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, tuned)", probs_directional_tuned,
    "GATv2 (unweighted, tuned)", probs_unweighted_tuned, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, tuned)", probs_directional_tuned,
    "GATv2 (old transform 1/(1+PAE), tuned)", test_df["old_transform_tuned_prob"].values, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, tuned)", probs_directional_tuned,
    "GraphSAGE (baseline)", test_df["graphsage_prob"].values, test_labels))
sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, default)", probs_directional_default,
    "GATv2 (directional PAE, tuned)", probs_directional_tuned, test_labels))

pd.DataFrame(sig_rows).to_csv(os.path.join(EXPERIMENT_DIR, "significance_tests.csv"), index=False)
print("\nSaved significance_tests.csv")


## 12. PAE Value Quantification (Proposal-Required Deliverable, This Variant)


In [ ]:
def pct_row(stage, pae_probs, unweighted_probs, labels, sig_row):
    pae_auprc = average_precision_score(labels, pae_probs)
    uw_auprc = average_precision_score(labels, unweighted_probs)
    pct = (pae_auprc - uw_auprc) / uw_auprc * 100
    return {"Stage": stage, "PAE-weighted AUPRC": round(pae_auprc, 4),
            "Unweighted AUPRC": round(uw_auprc, 4), "% change (PAE vs unweighted)": round(pct, 3),
            "bootstrap_p": round(sig_row["bootstrap_p"], 4), "delong_p": round(sig_row["delong_p"], 4),
            "significant (alpha=0.05)": bool(sig_row["significant"])}


pae_quant_rows = [
    pct_row("Default (untuned)", probs_directional_default, probs_unweighted_default, test_labels, sig_rows[0]),
    pct_row("Tuned (B_reg_small)", probs_directional_tuned, probs_unweighted_tuned, test_labels, sig_rows[1]),
]
pae_quant_df = pd.DataFrame(pae_quant_rows)
pae_quant_df.to_csv(os.path.join(EXPERIMENT_DIR, "pae_value_quantification_directional.csv"), index=False)
print(pae_quant_df.to_string(index=False))


## 13. AlphaMissense Head-to-Head Comparison


In [ ]:
AA3TO1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C",
    "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H", "Ile": "I",
    "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P",
    "Ser": "S", "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
}
test_df["ref_1"] = test_df["Ref_AA"].map(AA3TO1)
test_df["variant_key"] = test_df["ref_1"] + test_df["Pos"].astype(str) + test_df["Alt_AA_1"]

am_scores = pd.read_csv(AM_CACHE_PATH)
am_clean = am_scores.rename(columns={"uniprot_id": "UniProt_ID", "protein_variant": "variant_key"})[
    ["UniProt_ID", "variant_key", "am_pathogenicity", "am_class"]]

merged = test_df.merge(am_clean, on=["UniProt_ID", "variant_key"], how="inner")
coverage = len(merged) / len(test_df) * 100
print(f"Coverage: {len(merged):,} / {len(test_df):,} test variants matched ({coverage:.1f}%)")

AM_THRESHOLD = 0.564
am_labels = merged["true_label"].values.astype(int)

def metrics_row(name, probs, labels, threshold=0.5):
    m = full_metrics(labels, probs, threshold)
    m["Model"] = name
    m["Threshold"] = threshold
    return m

am_results = [
    metrics_row("GATv2 (directional PAE, tuned)", merged["directional_tuned_prob"].values, am_labels, 0.5),
    metrics_row("GraphSAGE (baseline)", merged["graphsage_prob"].values, am_labels, 0.5),
    metrics_row("AlphaMissense", merged["am_pathogenicity"].values, am_labels, AM_THRESHOLD),
]
am_results_df = pd.DataFrame(am_results)[["Model", "AUPRC", "AUROC", "Accuracy", "Precision", "Recall", "F1", "Threshold"]]
am_results_df = am_results_df.sort_values("AUPRC", ascending=False)
am_results_df.to_csv(os.path.join(EXPERIMENT_DIR, "alphamissense_comparison_results.csv"), index=False)
print(am_results_df.to_string(index=False))

am_sig_rows = []
am_sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, tuned)", merged["directional_tuned_prob"].values,
    "AlphaMissense", merged["am_pathogenicity"].values, am_labels))
am_sig_rows.append(run_all_significance_tests(
    "GATv2 (directional PAE, tuned)", merged["directional_tuned_prob"].values,
    "GraphSAGE (baseline)", merged["graphsage_prob"].values, am_labels))
pd.DataFrame(am_sig_rows).to_csv(os.path.join(EXPERIMENT_DIR, "alphamissense_significance_tests.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = ["GATv2\n(directional PAE,\ntuned)", "GraphSAGE\n(baseline)", "AlphaMissense"]
colors = ["#2E6F8E", "#4CAF50", "#FF5722"]
for ax, metric in zip(axes, ["AUPRC", "AUROC"]):
    vals = [am_results_df.set_index("Model").loc[n, metric] for n in
            ["GATv2 (directional PAE, tuned)", "GraphSAGE (baseline)", "AlphaMissense"]]
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor="white")
    ax.set_ylim(min(vals) - 0.02, max(vals) + 0.03)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} - matched subset (n={len(merged):,})")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001, f"{val:.4f}",
                ha="center", va="bottom", fontsize=9)
plt.suptitle("Directional PAE features vs. GraphSAGE vs. AlphaMissense", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(NEW_FIGURES_DIR, "alphamissense_comparison_directional.png"), dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)


## 14. Final Summary


In [ ]:
print("="*70)
print("SUMMARY: Directional PAE Edge Features vs. Original Symmetrized 1/(1+PAE)")
print("="*70)
print(f"\nModel comparison (full table in results_comparison.csv):")
print(results_comparison_df[["model", "AUPRC", "AUROC", "Accuracy"]].to_string(index=False))
print(f"\nPAE Value Quantification (full table in pae_value_quantification_directional.csv):")
print(pae_quant_df.to_string(index=False))
print(f"\nAlphaMissense comparison (full table in alphamissense_comparison_results.csv):")
print(am_results_df.to_string(index=False))


### Methods note (copy-paste for the paper's robustness-check / limitations section)
> As a second robustness check, the original PAE-weighting pipeline's directional averaging step (`PAE[i][j]` and `PAE[j][i]` collapsed to a single symmetrized scalar) was replaced with two separate directional edge features, increasing `edge_dim` from 2 to 3. A diagnostic on a 300-protein sample showed 32.8% of edges tied under the symmetrized transform were distinguishable once directional information was retained, while 67.2% remained genuinely tied (AlphaFold reports identical confidence in both directions for those pairs). The GATv2 architecture, training procedure, hyperparameter configurations, and train/val/test split were kept identical to the main experiment. [Fill in after running: whether recovering this directional information changed the PAE-weighted-vs-unweighted comparison's direction or significance.]
